In [1]:
%sql
select
    'CRE' as phenotype,
    ast_cre_status as status,
    count(*) as records
from workspace.micro_dev.int_culture_organism_phenotype
group by ast_cre_status

union all

select
    'ESCE-R',
    ast_esce_enterobacterales_status,
    count(*)
from workspace.micro_dev.int_culture_organism_phenotype
group by ast_esce_enterobacterales_status

union all

select
    'Oxacillin-resistant S. aureus',
    ast_oxacillin_resistance_status,
    count(*)
from workspace.micro_dev.int_culture_organism_phenotype
group by ast_oxacillin_resistance_status

union all

select
    'VRE',
    ast_vancomycin_resistance_status,
    count(*)
from workspace.micro_dev.int_culture_organism_phenotype
group by ast_vancomycin_resistance_status

union all

select
    'MDR P. aeruginosa',
    ast_mdr_p_aeruginosa_status,
    count(*)
from workspace.micro_dev.int_culture_organism_phenotype
group by ast_mdr_p_aeruginosa_status

order by phenotype, status

,phenotype,status,records
0,CRE,not_applicable,47231
1,CRE,not_detected,59910
2,CRE,not_evaluable,25905
3,CRE,positive,200
4,ESCE-R,not_applicable,56432
5,ESCE-R,not_detected,54196
6,ESCE-R,not_evaluable,15460
7,ESCE-R,positive,7158
8,MDR P. aeruginosa,not_applicable,122780
9,MDR P. aeruginosa,not_detected,6572


In [2]:
%sql
with source_model as (

    select
        count(*) as row_count,
        count(distinct culture_organism_key) as unique_keys

    from workspace.micro_dev.int_mapped_cohort_culture_organism

),

phenotype_model as (

    select
        count(*) as row_count,
        count(distinct culture_organism_key) as unique_keys

    from workspace.micro_dev.int_culture_organism_phenotype

)

select
    s.row_count as source_rows,
    s.unique_keys as source_unique_keys,

    p.row_count as phenotype_rows,
    p.unique_keys as phenotype_unique_keys,

    p.row_count - s.row_count as row_difference,

    case
        when s.row_count = p.row_count
         and s.unique_keys = p.unique_keys
         and p.row_count = p.unique_keys
            then 'pass'

        else 'fail'
    end as validation_status

from source_model s
cross join phenotype_model p

,source_rows,source_unique_keys,phenotype_rows,phenotype_unique_keys,row_difference,validation_status
0,133246,133246,133246,133246,0,pass


In [4]:
%sql
select
    reporting_phenotype,

    count(*) as reporting_records,

    sum(
        case
            when source_organism_record_count > 1
                then 1
            else 0
        end
    ) as multi_source_records,

    max(
        source_organism_record_count
    ) as max_source_records_per_reporting_record

from workspace.micro_dev.int_reporting_culture_organism

group by reporting_phenotype

order by reporting_phenotype

,reporting_phenotype,reporting_records,multi_source_records,max_source_records_per_reporting_record
0,base,100658,1272,2
1,cre,198,0,1
2,esce_r,6982,0,1
3,mdr_p_aeruginosa,993,82,2
4,oxacillin_resistant,2390,32,2
5,vancomycin_resistant,250,0,1


In [5]:
%sql
with p as (

    select *
    from workspace.micro_dev.int_culture_organism_phenotype

),

classified as (

    select
        *,

        case
            when include_primary_summary <> 1
              or mapping_status <> 'reviewed'
              or organism_reporting_group is null
                then 'excluded_from_reporting_layer'

            when ast_cre_status = 'positive'
                then 'cre'

            when ast_mdr_p_aeruginosa_status = 'positive'
                then 'mdr_p_aeruginosa'

            when ast_oxacillin_resistance_status = 'positive'
                then 'oxacillin_resistant'

            when ast_vancomycin_resistance_status = 'positive'
                then 'vancomycin_resistant'

            when ast_esce_enterobacterales_status = 'positive'
                then 'esce_r'

            else 'base'
        end as expected_reporting_phenotype

    from p

)

select
    'CRE positive source records' as check_name,

    count(*) as source_records,

    sum(
        case
            when expected_reporting_phenotype = 'cre'
                then 1
            else 0
        end
    ) as retained_in_category,

    sum(
        case
            when expected_reporting_phenotype =
                 'excluded_from_reporting_layer'
                then 1
            else 0
        end
    ) as excluded_from_reporting_layer,

    0 as reassigned_by_precedence

from classified

where ast_cre_status = 'positive'

union all

select
    'ESCE-R positive source records',

    count(*),

    sum(
        case
            when expected_reporting_phenotype = 'esce_r'
                then 1
            else 0
        end
    ),

    sum(
        case
            when expected_reporting_phenotype =
                 'excluded_from_reporting_layer'
                then 1
            else 0
        end
    ),

    sum(
        case
            when expected_reporting_phenotype = 'cre'
                then 1
            else 0
        end
    )

from classified

where ast_esce_enterobacterales_status = 'positive'

,check_name,source_records,retained_in_category,excluded_from_reporting_layer,reassigned_by_precedence
0,CRE positive source records,200,198,2,0
1,ESCE-R positive source records,7158,6982,2,174


In [6]:
%sql
with duplicates as (

    select
        source_version,
        cohort_id,
        patient_key,
        organism_reporting_group,
        reporting_phenotype,
        count(*) as records

    from workspace.micro_dev.int_first_reporting_isolate

    group by
        source_version,
        cohort_id,
        patient_key,
        organism_reporting_group,
        reporting_phenotype

    having count(*) > 1

)

select
    (
        select count(*)
        from workspace.micro_dev.int_first_reporting_isolate
    ) as selected_rows,

    (
        select count(distinct first_reporting_isolate_key)
        from workspace.micro_dev.int_first_reporting_isolate
    ) as unique_keys,

    (
        select count(*)
        from duplicates
    ) as duplicate_reporting_groups,

    case
        when (
            select count(*)
            from duplicates
        ) = 0

        and (
            select count(*)
            from workspace.micro_dev.int_first_reporting_isolate
        ) = (
            select count(distinct first_reporting_isolate_key)
            from workspace.micro_dev.int_first_reporting_isolate
        )

        then 'pass'
        else 'fail'
    end as validation_status

,selected_rows,unique_keys,duplicate_reporting_groups,validation_status
0,77369,77369,0,pass
